# Student Learning Analytics: EDA to MongoDB Pipeline

This notebook covers 6 stages end-to-end:

1. Setup: start Spark and load the dataset
2. Data validation: quick schema, nulls, duplicates checks
3. Exploratory analysis: categorical and numerical summaries
4. Text processing: tokenize, filter stopwords, aggregate frequencies
5. Sentiment analysis: compute VADER compound score and label
6. MongoDB storage: write processed collections for Tableau

We use `helpers.py` for reusable helper functions (tokenization, stopwords, word-frequency, token-bridge, and VADER scoring). Keeping those functions in a separate file makes the notebook concise and easier to maintain; the helpers are lightly commented to explain intent.

The notebook is structured to show the analysis steps first, then the pipeline export to MongoDB at the end.

## 1. Setup

In [ ]:
# Initialize a SparkSession for the notebook
# This session is reused across the cells below.
from pyspark.sql import SparkSession
from helpers import load_student_dataset

spark = SparkSession.builder \
    .appName("Student Learning Analytics — MongoDB Pipeline") \
    .getOrCreate()

In [ ]:
# Load the dataset using the shared helper to keep this cell minimal.
# The CSV path is workspace-relative.
df = load_student_dataset(
    spark,
    "sources/synthetic_student_learning_dataset_10000.csv",
)

## 2. Data validation

Inspect the inferred schema to confirm column names and data types.

In [3]:
df.printSchema()
df.show(5, truncate=False)

root
 |-- respondent_id: integer (nullable = true)
 |-- education_level: string (nullable = true)
 |-- study_hours_per_day: integer (nullable = true)
 |-- preferred_learning_method: string (nullable = true)
 |-- main_learning_challenge: string (nullable = true)
 |-- motivation_level: string (nullable = true)
 |-- online_learning_opinion: string (nullable = true)
 |-- device_used_for_study: string (nullable = true)

+-------------+---------------+-------------------+-------------------------+---------------------------------+----------------+------------------------------------------------------+---------------------+
|respondent_id|education_level|study_hours_per_day|preferred_learning_method|main_learning_challenge          |motivation_level|online_learning_opinion                               |device_used_for_study|
+-------------+---------------+-------------------+-------------------------+---------------------------------+----------------+-----------------------------------------

Confirm the number of rows and columns.

In [4]:
print("Rows   :", df.count())
print("Columns:", len(df.columns))

Rows   : 10000
Columns: 8


Check for null values in every column. A null-free dataset is expected here because the data was synthetically generated with controlled vocabularies.

In [5]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show()

+-------------+---------------+-------------------+-------------------------+-----------------------+----------------+-----------------------+---------------------+
|respondent_id|education_level|study_hours_per_day|preferred_learning_method|main_learning_challenge|motivation_level|online_learning_opinion|device_used_for_study|
+-------------+---------------+-------------------+-------------------------+-----------------------+----------------+-----------------------+---------------------+
|            0|              0|                  0|                        0|                      0|               0|                      0|                    0|
+-------------+---------------+-------------------+-------------------------+-----------------------+----------------+-----------------------+---------------------+



Check for fully duplicate rows.

In [6]:
duplicates = df.count() - df.dropDuplicates().count()
print("Duplicate rows:", duplicates)

Duplicate rows: 0


## 3. Exploratory analysis

### 3.1 Categorical distributions

Print distinct value counts for every categorical column. The dataset uses controlled vocabularies, so the number of distinct values per column is small and the distributions are near-uniform.

In [7]:
categorical_columns = [
    "education_level",
    "preferred_learning_method",
    "main_learning_challenge",
    "motivation_level",
    "device_used_for_study",
]

for c in categorical_columns:
    print(f"\n── {c} ──")
    df.groupBy(c) \
      .count() \
      .orderBy("count", ascending=False) \
      .show(truncate=False)


── education_level ──
+---------------+-----+
|education_level|count|
+---------------+-----+
|Graduate       |3368 |
|Postgraduate   |3320 |
|Undergraduate  |3312 |
+---------------+-----+


── preferred_learning_method ──
+-------------------------+-----+
|preferred_learning_method|count|
+-------------------------+-----+
|Interactive discussion   |2040 |
|Video lectures           |2001 |
|Practice exercises       |1998 |
|Reading notes            |1997 |
|Recorded tutorials       |1964 |
+-------------------------+-----+


── main_learning_challenge ──
+---------------------------------+-----+
|main_learning_challenge          |count|
+---------------------------------+-----+
|Time management difficulty       |1738 |
|Lack of concentration            |1716 |
|Internet connectivity issues     |1672 |
|Academic workload pressure       |1636 |
|Low motivation                   |1620 |
|Difficulty understanding concepts|1618 |
+---------------------------------+-----+


── motivation_l

### 3.2 Study hours distribution

Summary statistics for the only numerical field in the dataset.

In [8]:
df.select("study_hours_per_day").describe().show()

# Frequency distribution across all possible values (1–6)
df.groupBy("study_hours_per_day") \
  .count() \
  .orderBy("study_hours_per_day") \
  .show()

+-------+-------------------+
|summary|study_hours_per_day|
+-------+-------------------+
|  count|              10000|
|   mean|             3.4745|
| stddev| 1.7054737213048505|
|    min|                  1|
|    max|                  6|
+-------+-------------------+

+-------------------+-----+
|study_hours_per_day|count|
+-------------------+-----+
|                  1| 1706|
|                  2| 1668|
|                  3| 1687|
|                  4| 1676|
|                  5| 1640|
|                  6| 1623|
+-------------------+-----+



### 3.3 Education level × preferred learning method

A cross-tabulation to explore whether the two variables co-vary. Because the dataset is synthetic and correlations are near-zero by design, no meaningful pattern is expected.

In [9]:
df.groupBy("education_level", "preferred_learning_method") \
  .count() \
  .orderBy("education_level", "count", ascending=False) \
  .show(truncate=False)

+---------------+-------------------------+-----+
|education_level|preferred_learning_method|count|
+---------------+-------------------------+-----+
|Undergraduate  |Reading notes            |666  |
|Undergraduate  |Video lectures           |664  |
|Undergraduate  |Practice exercises       |663  |
|Undergraduate  |Recorded tutorials       |662  |
|Undergraduate  |Interactive discussion   |657  |
|Postgraduate   |Interactive discussion   |692  |
|Postgraduate   |Video lectures           |680  |
|Postgraduate   |Reading notes            |675  |
|Postgraduate   |Recorded tutorials       |650  |
|Postgraduate   |Practice exercises       |623  |
|Graduate       |Practice exercises       |712  |
|Graduate       |Interactive discussion   |691  |
|Graduate       |Video lectures           |657  |
|Graduate       |Reading notes            |656  |
|Graduate       |Recorded tutorials       |652  |
+---------------+-------------------------+-----+



## 4. Text processing pipeline

The helper from `helpers.py` builds the token table, filters stop words, aggregates word frequencies, and prepares the respondent-level token table. Sentiment is handled in the next section.

In [ ]:
# Build token tables and word-frequency using helpers.
# Returning these DataFrames keeps the main notebook readable.
from helpers import build_text_features

opinion_tokens, opinion_tokens_filtered, word_freq, processed_opinions = build_text_features(df)

## 5. Sentiment analysis

Apply VADER to each opinion. The compound score is computed with a PySpark UDF and then classified into three labels using the standard VADER thresholds.

In [ ]:
# Score sentiments using VADER (kept as a separate step for clarity)
from helpers import add_vader_sentiment

processed_opinions = add_vader_sentiment(processed_opinions)

# Show a sample of the scored opinions
processed_opinions.select(
    "respondent_id", "online_learning_opinion",
    "sentiment_score", "sentiment_label"
).show(truncate=False)

+-------------+------------------------------------------------------+---------------+---------------+
|respondent_id|online_learning_opinion                               |sentiment_score|sentiment_label|
+-------------+------------------------------------------------------+---------------+---------------+
|1            |Online learning is flexible and convenient            |0.2263         |positive       |
|2            |Online learning is effective for theory-based subjects|0.4767         |positive       |
|3            |Online learning is effective for theory-based subjects|0.4767         |positive       |
|4            |Online platforms provide access to diverse resources  |0.0            |neutral        |
|5            |Online learning is flexible and convenient            |0.2263         |positive       |
|6            |Blended learning is more effective than fully online  |0.5256         |positive       |
|7            |Online learning requires strong self-discipline       |0.5

## 6. MongoDB storage

Connect to MongoDB Atlas using an environment variable instead of embedding credentials in the notebook.

Before running this section, set `MONGO_URI` in your shell by using $env:MONGO_URI = "your-atlas-uri". The connection cell verifies TLS and runs a `ping` immediately so certificate/network issues appear early.

If TLS errors persist, check Atlas Network Access, and make sure you add you IP address to the cluster.

In [ ]:

from pymongo import MongoClient
from pymongo.server_api import ServerApi
import os

# Connect to MongoDB Atlas using a secured environment variable.
uri = os.environ.get("MONGO_URI")
# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [ ]:
# Access the "online_learning" database and its collections.
db = client["online_learning"]

processed_dataset_col = db["processed_dataset"]
word_count_col = db["word_count"]
token_bridge_col = db["opinion_token_bridge"]

### 6.1 `processed_dataset` collection

One document per respondent. The original `respondent_id` becomes MongoDB's built-in `_id`. Each document stores all survey fields plus the token array, sentiment score, and sentiment label.

Records are collected into a Python list and inserted in a single `insert_many` call, which is more efficient than inserting them one by one.

In [21]:
processed_dataset_col.delete_many({})

# Join the original survey fields with the processed opinion data
full_df = df.join(
    processed_opinions.select(
        "respondent_id", "tokens", "sentiment_score", "sentiment_label"
    ),
    on="respondent_id",
    how="left"
)

records = []
for row in full_df.collect():
    doc = row.asDict(recursive=True)
    doc["_id"] = doc.pop("respondent_id")
    records.append(doc)

if records:
    processed_dataset_col.insert_many(records)

print("Documents inserted:", processed_dataset_col.count_documents({}))

Documents inserted: 10000


Verify the structure of a sample document.

In [22]:
processed_dataset_col.find_one()

{'_id': 1,
 'education_level': 'Undergraduate',
 'study_hours_per_day': 6,
 'preferred_learning_method': 'Practice exercises',
 'main_learning_challenge': 'Time management difficulty',
 'motivation_level': 'Low',
 'online_learning_opinion': 'Online learning is flexible and convenient',
 'device_used_for_study': 'Tablet',
 'tokens': ['convenient', 'flexible'],
 'sentiment_score': 0.22630000114440918,
 'sentiment_label': 'positive'}

### 6.2 `word_count` collection

One document per meaningful term with its opinion count and percentage.
This collection powers the keyword distribution chart in Tableau.

In [23]:
word_count_col.delete_many({})

wc_records = []
for row in word_freq.collect():
    doc = row.asDict(recursive=True)
    doc["word"] = doc.pop("token")
    wc_records.append(doc)

if wc_records:
    word_count_col.insert_many(wc_records)

print("Documents inserted:", word_count_col.count_documents({}))

Documents inserted: 15


Verify the structure of a sample document.

In [24]:
word_count_col.find_one()

{'_id': ObjectId('6a173d549d1986100bac9447'),
 'opinion_count': 3975,
 'pct_opinions': 39.75,
 'word': 'effective'}

### 6.3 `opinion_token_bridge` collection

One document per respondent-token pair. Created by exploding the token arrays from `processed_opinions`. This flat structure lets Tableau join on `respondent_id` and filter the full respondent view by any individual token without needing to parse array fields directly.

In [ ]:
from helpers import build_token_bridge_df


token_bridge_df = build_token_bridge_df(processed_opinions)

bridge_records = [row.asDict(recursive=True) for row in token_bridge_df.collect()]

if bridge_records:
    token_bridge_col.insert_many(bridge_records)

print("Documents inserted:", token_bridge_col.count_documents({}))

Documents inserted: 32206


Verify the structure of a sample document and confirm document counts.

In [26]:
for doc in token_bridge_col.find({}, {"_id": 0}).limit(5):
    print(doc)

{'respondent_id': 1, 'token': 'convenient'}
{'respondent_id': 1, 'token': 'flexible'}
{'respondent_id': 2, 'token': 'effective'}
{'respondent_id': 2, 'token': 'subjects'}
{'respondent_id': 2, 'token': 'theory-based'}


In [27]:
print("processed_dataset   :", processed_dataset_col.count_documents({}))
print("word_count          :", word_count_col.count_documents({}))
print("opinion_token_bridge:", token_bridge_col.count_documents({}))

processed_dataset   : 10000
word_count          : 15
opinion_token_bridge: 32206


## 7. Close connections

We close the MongoDB client and stop the Spark session to release resources.

In [28]:
client.close()
spark.stop()